# Assignment 2



## Question 1

A nuclear fuel pellet is a cylinder, 1.5 cm in lenth and 1 cm in diameter. Assume the surface temperature is 300 C everywhere. Given temperature probe data below, determine the radial temperature profile in the middle of a nuclear fuel pellet (i.e.: T(r, z = 0.75)) using radial basis functions.


In [ ]:
import numpy as np

# 20 data points presented in columns: | x | y | z | T |, measurements in cm

data = np.array([
    [5.1690e-02, 2.3766e-01, 6.7059e-01, 5.2645e+02],
    [1.1353e-01, 9.4708e-02, 5.3856e-01, 5.5201e+02],
    [1.6676e-01, 1.4358e-01, 4.6936e-01, 5.0802e+02],
    [1.3610e-01, 3.7207e-02, 2.1694e-01, 4.3663e+02],
    [8.9225e-02, 3.7293e-01, 1.1270e+00, 3.9234e+02],
    [1.9001e-01, 3.7240e-01, 8.4774e-01, 3.8872e+02],
    [5.4849e-02, 3.5425e-01, 5.7478e-01, 4.3784e+02],
    [1.7001e-01, 2.0241e-01, 1.2960e+00, 4.0159e+02],
    [2.0606e-01, 3.1594e-01, 6.4077e-01, 4.2652e+02],
    [2.5382e-01, 2.5859e-01, 4.8610e-01, 4.2481e+02],
    [5.6038e-02, 8.2231e-02, 4.2029e-01, 5.3244e+02],
    [3.1242e-01, 8.0489e-02, 1.1530e+00, 4.2453e+02],
    [6.0186e-02, 4.4891e-01, 3.9941e-01, 3.4207e+02],
    [1.5070e-01, 3.4794e-01, 1.5595e-01, 3.4750e+02],
    [1.8215e-01, 3.4388e-01, 1.0478e+00, 3.9963e+02],
    [1.1633e-01, 4.1011e-01, 5.5001e-01, 3.7611e+02],
    [1.2377e-01, 3.3703e-01, 3.7672e-02, 3.1423e+02],
    [4.6378e-02, 3.3653e-01, 1.4434e+00, 3.2345e+02],
    [2.9063e-02, 3.2584e-02, 2.3977e-01, 4.5993e+02],
    [2.1162e-02, 3.8590e-01, 2.5905e-01, 3.6901e+02]
])

Consider what you know about this system. What extra information do you have in terms of

### a) type(s) of symmetry?

Assuming the pellet's material and heat generation are uniform, we would have two types of symmetry. Rotational and midplane.  

Rotational: Rotating around the cylinder's axis doesn't change the temperature so we can use r and z, rather than x, y, and z to describe temperature.

Midplane symmetry: Positions equally far above and below z=0.75 have the same temperatures. This can be described by the following:

T(r, z)=T(r, 1.5-z)

Where r is the distance outward from the cylinder's central axis (0 to 0.5cm), and z is the distance along the clinders length (0 to 1.5cm)

### b) Boundary conditions?

The entire outer surface of the cylinder is 300 degrees celcius meaning that:

T(0.5, z) = 300 - The cylinder's wall
T(r, 0) = 300 - Bottom of cylinder
T(r, 1.5) = 300 - Top of cylinder

## c) Plot the best guess of the radial temperature profile

Note: RBFs will fail with a linear solver error if two data points exactly overlap.

{Method, implementation, answer, answer}

In [2]:
#Take the x position from every row in data, and store them in x
x = data[:, 0] # : means all rows, and 0 means the first column (x, pos)
y = data[:, 1]
z = data[:, 2]
T = data[:, 3]

#Uses pythagorean theorem to find how far each probe is from the
# cylinder's central axis

r = np.sqrt(x**2 + y**2)

#Next well add the exterior wall surface T of 300 degrees, which aren't
# included in the probe data

# Creates 7 equally spaced points starting from 0 to 1.5
z_wall = np.linspace(0, 1.5, 7)

#Creates an array the same size as z_wall, filled with 0.5
# because every wall point has that radius
r_wall = np.full_like(z_wall, 0.5)

## Creates an array the same size as z_wall, filled with 300
# because every point on the exterior has that temperature
T_wall = np.full_like(z_wall, 300)

#Next we add the points on the flat bottom and top of the cylinder
# which are also at 300 degrees
# Here z is fixed while r changes from the central axis,
# towards the cylinder's wall

#Creates 5 equally spaced points starting from 0, t0 0.5
# endpoint = false, leaves out r = 0.5 as its already included
# where the top and bottom meet the wall
r_ends = np.linespace(0, 0.5, 5, endpoint = False)

#Creates an array the same size as r_ends filled with all 0
# because thats the height of the bottom of the cylinder
z_bottom = np.full_like(r_ends, 0)

#Creates an arrray the same size as r_ends filled with all 1.5
# because thats the height of the top of the cylinder
z_top = np.full_like(r_ends, 1.5)

#Creates array same size as r_ends filled with 300
# cause thats the temp at both the top and bottom exterior
T_ends = np.full_like(r_ends, 300)

# NEXT, combine probe data and surface points into 1 dataset

#np.concatenate joins arrays end to end. For ex: [1,2] and [3,4]
# gives [1,2,3,4]

#Order of the data: Probes (r, z, t), wall, bottom, top

# we repeat r_ends cause the top and bottom have same radial position
# but their diff heights make them diff locations
r_all = np.concatenate((r, r_wall, r_ends, r_ends))
z_all = np.concatenate((z, z_wall, z_bottom, z_top))
T_all = np.concatenate((T, T_wall, T_ends, T_ends))

# NEXT, define Gaussian RBF

# def creates a function named gaussian_rbf
# distance is the distance between 2 points in our (r, z) coordinates
# epsilon conrols how narror or wide the shape of the graph is (the bell)
# np.exp calculates e raised to the expression inside
# return send the calculated value back
def gaussian_rbf(distance, epsilon):
  return np.exp(-(epsilon * distance)**2)

#NEXT, calculate the distance between every pair of known points

# This counts all of our data points in r_all
n = len(r_all)

#Creats a matrix the size of our data points and fills it with 0's
np.zeros((n,n))

#i selects one point and j runs through every point to
# compare with it
for i in range(n):
  for j in range(n):

  #stores the distance between points i and j
  dist_matrix[i,j] = np.sqrt(
      (r_all[i] - r_all[j]**2 + z_all[i] - z_all[j]**2)

  )


#NEXT, choose epsilon and apply the gaussian function to our distances

#calculates the average of all entriers in our distance
average_distance = np.mean(dist_matrix)

#controls how quickly the gaussian value decreases as distance increase
epsilon = 1 / average_distance

#Passes every distance through the gaussian equation we defined
# earlier
phi_matrix = gaussian_rbf(dist_matrix, epsilon)

#NEXT, find how much to multiply each gaussian by (their weights)
# to match their known temperatures

#phi_matrix contains the gaussian values we calculated
#T_all contains the temperatures we want to match
#np.linalg.solve finds all the point's weights that
# makes those equations hold
weights = np.linalg.solve(phi_matrix, T_all)

#NEXT, choose 100 points from the central axis to the wall
# at the middle height

#Contains 100 evenly spaced radial positions from 0 to 0.5
r_plot = np.linspace(0, 0.5, 100)
#Contains 100 copies of 0.75, keeping every point at the
# middle height
z_plot = np.full_like(r_plot, 0.75)

#NEXT, calculate the temperatures and plot them:
T_plot = np.zeros_like(r_plot)

for i in range(n):
  distance = np.sqrt(
      (r_plot - r_all[i]**2) + (z_plot - z_all[i]**2)

  )

  T_plot += weights[i] * gaussian_rbf(distance, epsilon)

#NEXT, Plot the graph

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=r_plot, y=T_plot, mode='lines'))

fig.update_layout(
    title='Radial Temperature Profile at z = 0.75 cm',
    xaxis_title='Radius r (cm)',
    yaxis_title='Temperature (°C)'
)

fig.show()


IndentationError: expected an indented block after 'for' statement on line 82 (3650314045.py, line 85)

# Question 2

You run an experiment and obtain the following data:

| x | y1 | y2 | y3 | y4 | y5 |
|---|---|---|---| --- | --- |
| 0.00 | -29.49 | -2.14 | 15.88 | 22.69 | 28.53 |
| 1.11 | 2.83 | 18.02 | -25.45 | -32.45 | 7.50 |
| 2.22 | 1.97 | -10.49 | -0.18 | -32.10 | -40.31 |
| 3.33 | -38.09 | -46.16 | -7.87 | -33.97 | -38.39 |
| 4.44 | -3.97 | -32.22 | -33.95 | -11.07 | -32.47 |
| 5.56 | 4.45 | -10.88 | 20.43 | 6.57 | -8.49 |
| 6.67 | 50.22 | 51.29 | 80.02 | 66.15 | 84.90 |
| 7.78 | 164.11 | 190.26 | 160.94 | 182.35 | 163.18 |
| 8.89 | 331.75 | 306.51 | 278.40 | 302.13 | 335.44 |
| 10.00 | 517.06 | 483.20 | 476.73 | 512.16 | 500.64 |



In [ ]:
import numpy as np

# Define the table as a list of lists
d = np.array([
    [0.00, -29.49, -2.14, 15.88, 22.69, 28.53],
    [1.11, 2.83, 18.02, -25.45, -32.45, 7.50],
    [2.22, 1.97, -10.49, -0.18, -32.10, -40.31],
    [3.33, -38.09, -46.16, -7.87, -33.97, -38.39],
    [4.44, -3.97, -32.22, -33.95, -11.07, -32.47],
    [5.56, 4.45, -10.88, 20.43, 6.57, -8.49],
    [6.67, 50.22, 51.29, 80.02, 66.15, 84.90],
    [7.78, 164.11, 190.26, 160.94, 182.35, 163.18],
    [8.89, 331.75, 306.51, 278.40, 302.13, 335.44],
    [10.00, 517.06, 483.20, 476.73, 512.16, 500.64]
])

## a) Determine the best cubic polynomial fit to this data with the uncertainty

{method, implementation, answer}

## b) Your manager thinks this should be a quadratic. Which do you think it should be and why?

{Answer}